# Customer Segmentation for Retail
### B.Tech II Year Minor Project in Machine Learning / Unsupervised Learning
**Project Title:** Customer Segmentation for Retail  
**Methodology:** RFM Feature Engineering + K-Means Clustering + Hierarchical Clustering + PCA  
**Dataset:** UCI Machine Learning Repository — Online Retail Dataset


## 1. Project Motivation and Pipeline Overview
Customer segmentation allows retailers to partition diverse customer bases into distinct groups exhibiting similar shopping behaviors.

The complete ML pipeline:
1. **Raw Data Ingestion & Audit:** UCI Online Retail transaction logs (541,909 records).
2. **Data Cleaning Pipeline:** Removal of missing CustomerIDs, cancellation invoices (`C*`), invalid quantities/prices, and duplicate records.
3. **RFM Feature Engineering:** Aggregating transactions into customer-level Recency ($R$), Frequency ($F$), and Monetary ($M$) dimensions.
4. **Skewness & Variance Normalization:** Logarithmic transform (`np.log1p`) followed by `StandardScaler` to ensure zero-mean unit-variance.
5. **Cluster Selection & Evaluation:** Mathematical optimization via Elbow Method (Inertia) and Silhouette Analysis across $K \in [2, 8]$.
6. **K-Means Clustering:** Core partitional clustering model.
7. **Hierarchical Clustering Benchmark:** Agglomerative clustering with Ward linkage.
8. **Dimensionality Reduction (PCA 2D):** Visualizing high-dimensional cluster separation.
9. **Behavioral Personas & Actionable Retail Strategies:** Deriving business insights for customer retention and lifetime value maximization.


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualizations
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

# Import modular project pipeline
sys.path.append('..')
from src.data_loader import load_raw_data
from src.data_preprocessing import clean_transactions
from src.rfm_analysis import calculate_rfm, get_rfm_summary_stats, get_rfm_correlations
from src.clustering import preprocess_rfm_for_ml, evaluate_k_range, fit_kmeans, fit_hierarchical, compute_pca_2d
from src.evaluation import compare_clustering_algorithms, compute_cluster_cross_tab
from src.profiling import generate_cluster_profiles


## 2. Data Loading & Cleaning Audit Trail
The UCI dataset contains transaction logs. Unsupervised customer modeling requires rigorous cleaning before customer aggregation.


In [ ]:
# Load raw transactions
df_raw = load_raw_data('../data/online_retail.csv')
print(f'Raw dataset loaded successfully with shape: {df_raw.shape}')
df_raw.head()


In [ ]:
# Run comprehensive data cleaning pipeline
df_clean, audit = clean_transactions(df_raw)

print('=== DATA CLEANING AUDIT SUMMARY ===')
print(f'Initial Raw Transactions:    {audit["initial_rows"]:,}')
print(f'Missing CustomerIDs Removed: -{audit["initial_missing_customer"]:,}')
print(f'Cancelled Invoices Removed:  -{audit["cancelled_count"]:,}')
print(f'Invalid Price/Qty Records:   -{audit["invalid_qty_count"] + audit["invalid_price_count"]:,}')
print(f'Duplicate Records Dropped:   -{audit["duplicate_count"]:,}')
print(f'Clean Transactions Retained: {audit["final_rows"]:,} ({audit["retention_rate_pct"]}%)')
print(f'Unique Active Customers:     {audit["unique_customers"]:,}')
print(f'Unique Invoices:             {audit["unique_invoices"]:,}')
print(f'Total Gross Revenue:         £{audit["total_revenue"]:,.2f}')


## 3. Customer RFM Feature Engineering
We transform transactional logs into behavioral customer vectors:
* **Recency ($R$):** Days since customer's last transaction relative to the reference snapshot date (`max(date) + 1`).
* **Frequency ($F$):** Number of unique transactions (`InvoiceNo`) placed.
* **Monetary ($M$):** Total monetary expenditure across all purchases.


In [ ]:
# Extract RFM features
rfm_df, snapshot_date = calculate_rfm(df_clean)
print(f'Calculated snapshot reference date: {snapshot_date.strftime("%Y-%m-%d")}')
rfm_df.head(10)


In [ ]:
# Summary statistics and skewness analysis
rfm_summary = get_rfm_summary_stats(rfm_df)
rfm_summary


In [ ]:
# Visualize raw RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

sns.histplot(rfm_df['Recency'], kde=True, ax=axes[0], color='royalblue', bins=30)
axes[0].set_title('Recency Distribution (Days)', fontweight='bold')
axes[0].set_xlabel('Days Since Last Purchase')

sns.histplot(rfm_df['Frequency'], kde=True, ax=axes[1], color='forestgreen', bins=30)
axes[1].set_title('Frequency Distribution (Orders)', fontweight='bold')
axes[1].set_xlim(0, rfm_df['Frequency'].quantile(0.98))

p98 = rfm_df['Monetary'].quantile(0.98)
sns.histplot(rfm_df['Monetary'][rfm_df['Monetary'] <= p98], kde=True, ax=axes[2], color='darkorange', bins=30)
axes[2].set_title('Monetary Distribution (£ Spend)', fontweight='bold')

plt.tight_layout()
plt.show()


## 4. Feature Preprocessing for Machine Learning
Retail RFM features exhibit substantial positive skewness (Frequency skewness = 12.07, Monetary skewness = 19.34).
Clustering on raw values violates distance assumptions and causes Monetary scale to overshadow Recency and Frequency.
We apply:
1. **$x' = \ln(x + 1)$ Transformation** to compress heavy tails.
2. **StandardScaler** to map features to mean = 0 and unit standard deviation ($\sigma = 1$).


In [ ]:
scaled_matrix, scaled_df, scaler = preprocess_rfm_for_ml(rfm_df)
print('Scaled Features Sample:')
scaled_df.head()


## 5. Cluster Selection: Elbow Method & Silhouette Analysis
We iterate $K \in [2, 8]$ to evaluate optimal clustering geometry.


In [ ]:
eval_df = evaluate_k_range(scaled_matrix, k_range=range(2, 9))
eval_df


In [ ]:
# Plot Elbow Curve and Silhouette Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(eval_df['K'], eval_df['Inertia'], marker='o', color='navy', linewidth=2)
axes[0].axvline(x=4, color='crimson', linestyle='--', label='Selected K = 4')
axes[0].set_title('Elbow Method: Inertia vs K', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].legend()

axes[1].plot(eval_df['K'], eval_df['Silhouette_Score'], marker='s', color='seagreen', linewidth=2)
axes[1].axvline(x=4, color='crimson', linestyle='--', label='Selected K = 4')
axes[1].set_title('Silhouette Analysis: Score vs K', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Model Training: K-Means Clustering ($K=4$)
We train K-Means with $K=4$, generating cluster profiles and computing aggregate business statistics.


In [ ]:
K_CHOSEN = 4
kmeans_model, kmeans_labels = fit_kmeans(scaled_matrix, k=K_CHOSEN)
profile_df, personas = generate_cluster_profiles(rfm_df, kmeans_labels)
profile_df


## 7. Algorithm Comparison: K-Means vs Agglomerative Hierarchical
We benchmark K-Means against Agglomerative Hierarchical Clustering using Ward linkage.


In [ ]:
agg_model, agg_labels = fit_hierarchical(scaled_matrix, k=K_CHOSEN)
comp_df = compare_clustering_algorithms(scaled_matrix, kmeans_labels, agg_labels, k=K_CHOSEN)
comp_df


In [ ]:
# Cross-tabulation comparison matrix
cross_tab = compute_cluster_cross_tab(kmeans_labels, agg_labels)
print('Cross-Tabulation Matrix (K-Means vs Hierarchical):')
cross_tab


## 8. Principal Component Analysis (2D Projection)
Projecting scaled 3D RFM features into 2 principal components to inspect cluster boundaries.


In [ ]:
pca_coords, pca_model = compute_pca_2d(scaled_matrix)
var1 = pca_model.explained_variance_ratio_[0] * 100
var2 = pca_model.explained_variance_ratio_[1] * 100
print(f'PC1 Explained Variance: {var1:.2f}% | PC2 Explained Variance: {var2:.2f}% | Total: {var1+var2:.2f}%')

df_pca = pd.DataFrame({
    'PC1': pca_coords[:, 0],
    'PC2': pca_coords[:, 1],
    'Cluster': kmeans_labels
})

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_pca,
    x='PC1',
    y='PC2',
    hue='Cluster',
    palette='tab10',
    alpha=0.6,
    s=30
)
plt.title(f'2D PCA Projection of Customer Clusters (Total Variance: {var1+var2:.1f}%)', fontsize=12, fontweight='bold')
plt.xlabel(f'PC1 ({var1:.1f}% variance)')
plt.ylabel(f'PC2 ({var2:.1f}% variance)')
plt.show()


## 9. Customer Personas and Actionable Retail Strategies
Translating cluster metrics into actionable commercial strategies for the retail enterprise.


In [ ]:
for c_id, p in personas.items():
    m = p['metrics']
    print(f'=== CLUSTER {c_id}: {p["name"]} ===')
    print(f'Priority:     {p["priority"]}')
    print(f'Customer Base: {m["customers"]:,} ({m["share_pct"]}%) | {m["revenue_pct"]}% of total revenue')
    print(f'Average RFM:  Recency {m["recency_mean"]} days | Frequency {m["frequency_mean"]} orders | Spend £{m["monetary_mean"]:,.2f}')
    print(f'Profile:      {p["description"]}')
    print(f'Action Plan:  {p["strategy"]}\n')
